# Tahap 2d — Sensus emoji/token, cek R², dan cek teks hampir-duplikat
**CPU biasa, ±5 menit. Tidak ada pelatihan.** Hasil disimpan ke `stage2/stage2d_*.csv`. Bagian: (A) berapa teks yang benar-benar memuat emoji, [EMO:..], [TEGAS]; (B) rekonsiliasi R² vs penurunan MSE; (C) cek hampir-duplikat train↔test dan kinerja tanpa teks yang mirip.

In [ ]:
import os, json, re, random, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'
DATA4=f'{DRIVE}/JURNAL-DATASET'
OUT4=f'{DRIVE}/JURNAL-OUTPUTS/laptop_jurnal'
PREP4=f'{OUT4}/prep'; PRED4=f'{OUT4}/preds'; S2=f'{OUT4}/stage2'
os.makedirs(S2,exist_ok=True)
def P4(fn):
    p=f'{DATA4}/{fn}'
    return p if os.path.exists(p) else f'{DRIVE}/tesis-dataset 3/{fn}'
CATEGORIES=['Intrusive','Avoidance','Arousal','NegCogMood']
SEED=42; random.seed(SEED); np.random.seed(SEED)
TAG='jurnal_final'
print('setup OK')

In [ ]:
norm=lambda s: re.sub(r'\s+',' ',str(s).lower().strip())
y_va=np.load(f'{PREP4}/y_va_v3.npy'); y_te=np.load(f'{PREP4}/y_te_v3.npy'); y_gold=np.load(f'{PREP4}/y_gold_v3.npy')
idx_tr=np.load(f'{PREP4}/idx_tr_all_v3.npy'); idx_va=np.load(f'{PREP4}/idx_va_v3.npy'); idx_ho=np.load(f'{PREP4}/idx_ho_v3.npy')
texts_all=np.load(f'{PREP4}/texts_all_v3.npy',allow_pickle=True).tolist()
set_test=set(norm(texts_all[i]) for i in idx_ho); set_val=set(norm(texts_all[i]) for i in idx_va)
mask_tr=[i for i in idx_tr if norm(texts_all[i]) not in set_test|set_val]
df_gold=pd.read_csv(f'{DATA4}/data_ahli_FIXED.csv',encoding='utf-8-sig')
df_nat=pd.read_csv(f'{DATA4}/anotasi_natural_2000.csv',encoding='utf-8-sig')
df_b3=pd.read_csv(f'{DATA4}/gold_batch3_577.csv',encoding='utf-8-sig')
df_b4=pd.read_csv(P4('gold_batch4_600.csv'),encoding='utf-8-sig')
df_b5=pd.read_csv(P4('gold_batch5_600.csv'),encoding='utf-8-sig')
nat_norm=df_nat['Segmen_teks'].map(norm); mask_nat=~nat_norm.isin(set_test)
b3_norm=df_b3['Segmen_teks'].map(norm)
mask_b3=~(b3_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b3_norm.isin(set(nat_norm[mask_nat])))
b4_norm=df_b4['Segmen_teks'].map(norm)
mask_b4=~(b4_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b4_norm.isin(set(nat_norm[mask_nat]))|b4_norm.isin(set(b3_norm[mask_b3])))
b5_norm=df_b5['Segmen_teks'].map(norm)
mask_b5=~(b5_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b5_norm.isin(set(nat_norm[mask_nat]))|b5_norm.isin(set(b3_norm[mask_b3]))|b5_norm.isin(set(b4_norm[mask_b4])))
tr_raw=df_gold['Segmen_teks'].iloc[mask_tr].tolist()+df_nat['Segmen_teks'][mask_nat].tolist()+df_b3['Segmen_teks'][mask_b3].tolist()+df_b4['Segmen_teks'][mask_b4].tolist()+df_b5['Segmen_teks'][mask_b5].tolist()
va_raw=df_gold['Segmen_teks'].iloc[idx_va].tolist(); te_raw=df_gold['Segmen_teks'].iloc[idx_ho].tolist()
Y_tr=np.vstack([y_gold[mask_tr],df_nat.loc[mask_nat,CATEGORIES].values.astype(float),df_b3.loc[mask_b3,CATEGORIES].values.astype(float),
                df_b4.loc[mask_b4,CATEGORIES].values.astype(float),df_b5.loc[mask_b5,CATEGORIES].values.astype(float)])
assert len(tr_raw)==5313==len(Y_tr), (len(tr_raw),len(Y_tr))
assert len(va_raw)==135 and len(te_raw)==271
print('train',len(tr_raw),'val',len(va_raw),'test',len(te_raw))

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, mean_squared_error, mean_absolute_error, accuracy_score
from scipy.stats import pearsonr
def binar(y): return (np.asarray(y)>=0.5).astype(int)
def tune_thr(Yv,Pv):
    grid=np.round(np.arange(0.30,0.701,0.01),2); thr=[]
    for k in range(4):
        sc=[f1_score(binar(Yv)[:,k],(Pv[:,k]>=t).astype(int),zero_division=0) for t in grid]
        thr.append(float(grid[int(np.argmax(sc))]))
    return thr
def evaluate(Yt,Pt,thr):
    Yb=binar(Yt); Pb=(Pt>=np.array(thr)).astype(int)
    auc=[roc_auc_score(Yb[:,k],Pt[:,k]) for k in range(4)]
    dom_ok=(np.argmax(Pt,1)==np.argmax(np.asarray(Yt),1)).mean()
    return {'F1_weighted':f1_score(Yb,Pb,average='weighted',zero_division=0),'F1_macro':f1_score(Yb,Pb,average='macro',zero_division=0),
            'AUC_mean':float(np.mean(auc)),'MSE':mean_squared_error(Yt,Pt),'MAE':mean_absolute_error(Yt,Pt),
            'Pearson_r':pearsonr(np.asarray(Yt).ravel(),np.asarray(Pt).ravel())[0],'Dominant_acc':dom_ok,
            **{f'AUC_{c}':a for c,a in zip(CATEGORIES,auc)}}
# baseline trivial semua-positif (sanity check)
triv=evaluate(y_te,np.ones_like(y_te,dtype=float),[0.5]*4)
print('Trivial all-positive  F1w=%.4f  F1m=%.4f'%(triv['F1_weighted'],triv['F1_macro']))

In [ ]:
import ast, glob
names={'TFIDF_Ridge':'TFIDF-Ridge','TFIDF_LinearSVR':'TFIDF-LinearSVR','TFIDF_LogReg':'TFIDF-LogReg','TFIDF_SVD_RBFSVR':'TFIDF-SVD-RBFSVR','BERT_only_IndoBERT_':'BERT-only','Hybrid_BERT_SVR':'Hybrid'}
TEST={}
for k,v in names.items():
    f=f'{S2}/test_pred_{k}.npy'
    if os.path.exists(f): TEST[v]=np.load(f)
    else: print('TIDAK ADA:',f,'| ada:',[os.path.basename(x) for x in glob.glob(S2+'/test_pred_*.npy')])
tab20=pd.read_csv(f'{S2}/stage2_baseline_table.csv',index_col=0)
THR={}
for m,v in zip(['TFIDF-Ridge','TFIDF-LinearSVR','TFIDF-LogReg','TFIDF-SVD-RBFSVR','BERT-only (IndoBERT)','Hybrid BERT-SVR'],list(names.values())[:6]):
    if m in tab20.index: THR[v]=[float(x) for x in re.findall(r'\d+\.\d+|\d+',str(tab20.loc[m,'Thresholds']).replace('np.float64',''))][:4]
print(list(TEST), THR)

## A. Sensus emoji dan token

In [ ]:
import re, collections
EMO=re.compile('[\U0001F000-\U0001FAFF\u2600-\u27BF\u2764\u2B50\u2B55\u00A9\u00AE\u203C\u2049]')
rows=[]
for nm,T in (('train',tr_raw),('val',va_raw),('test',te_raw)):
    n=len(T)
    rows.append({'set':nm,'n':n,'emoji_texts':sum(bool(EMO.search(t)) for t in T),'EMO_token_texts':sum('[EMO' in t for t in T),
                 'TEGAS_token_texts':sum('[TEGAS' in t for t in T),'repeated_punct_texts':sum(bool(re.search(r'([!?.…])\1{1,}',t)) for t in T)})
cen=pd.DataFrame(rows); print(cen.to_string(index=False)); cen.to_csv(f'{S2}/stage2d_emoji_census.csv',index=False)
cnt=collections.Counter(ch for t in tr_raw for ch in EMO.findall(t)); print('emoji terbanyak:',cnt.most_common(10))
print('\nContoh 3 teks pertama (potong 150 karakter):'); [print(' -',t[:150].replace('\n',' ')) for t in tr_raw[:3]]

## B. Rekonsiliasi R² dengan penurunan MSE (Hybrid dan BERT-only)

In [ ]:
from sklearn.metrics import r2_score
yt=np.asarray(y_te); mu=np.asarray(Y_tr).mean(0); Pm=np.tile(mu,(len(yt),1)); mse_base=((yt-Pm)**2).mean(); rows=[]
for m in ('Hybrid','BERT-only'):
    P=TEST[m]; per=[r2_score(yt[:,k],P[:,k]) for k in range(4)]
    rows.append({'Model':m,'MSE':((yt-P)**2).mean(),'R2_sklearn_uniform_avg':r2_score(yt,P),'R2_mean_of_per_category':np.mean(per),
                 **{f'R2_{c}':v for c,v in zip(CATEGORIES,per)},'R2_pooled_vs_test_mean':1-((yt-P)**2).sum()/((yt-yt.mean(0))**2).sum(),
                 'Reduction_vs_train_mean_pred':1-((yt-P)**2).mean()/mse_base})
r2=pd.DataFrame(rows); print(r2.round(4).T.to_string()); print('MSE prediktor rata-rata latih:',round(mse_base,4)); r2.to_csv(f'{S2}/stage2d_r2_check.csv',index=False)

## C. Cek teks hampir-duplikat antara test dan train (cosine char n-gram)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, roc_auc_score, matthews_corrcoef
v=TfidfVectorizer(lowercase=True,analyzer='char_wb',ngram_range=(3,5),min_df=2,sublinear_tf=True)
Xtr=v.fit_transform(tr_raw); Xte=v.transform(te_raw); Xva=v.transform(va_raw)
mx_te=np.asarray((Xte@Xtr.T).max(axis=1).todense()).ravel(); mx_va=np.asarray((Xva@Xtr.T).max(axis=1).todense()).ravel()
rows=[]
for th in (0.7,0.8,0.9,0.95):
    rows.append({'threshold':th,'test_near_dup':int((mx_te>=th).sum()),'test_pct':(mx_te>=th).mean()*100,'val_near_dup':int((mx_va>=th).sum()),'val_pct':(mx_va>=th).mean()*100})
nd=pd.DataFrame(rows); print(nd.round(2).to_string(index=False)); nd.to_csv(f'{S2}/stage2d_neardup_counts.csv',index=False)
print('median/maks kemiripan test->train:',round(float(np.median(mx_te)),3),round(float(mx_te.max()),3))

In [ ]:
def sub_metrics(m,idx):
    Yb=binar(y_te)[idx]; P=TEST[m][idx]; thr=np.array(THR[m]); Pb=(P>=thr).astype(int); yt_=np.asarray(y_te)[idx]
    mcc=np.mean([matthews_corrcoef(Yb[:,k],Pb[:,k]) if len(set(Pb[:,k]))>1 else 0 for k in range(4)])
    auc=np.mean([roc_auc_score(Yb[:,k],P[:,k]) for k in range(4)])
    return {'n':len(idx),'F1_weighted':f1_score(Yb,Pb,average='weighted',zero_division=0),'MCC_mean':mcc,'AUC_mean':auc,'MSE':((yt_-P)**2).mean()}
rows=[]
for th in (0.8,0.9):
    keep=np.where(mx_te<th)[0]
    for m in ('Hybrid','BERT-only','TFIDF-Ridge'):
        for lab,idx in (('semua',np.arange(len(y_te))),(f'tanpa_dup>={th}',keep)):
            if len(idx)>0: rows.append({'Model':m,'Subset':lab,**sub_metrics(m,idx)})
sm=pd.DataFrame(rows).drop_duplicates(['Model','Subset']); print(sm.round(4).to_string(index=False)); sm.to_csv(f'{S2}/stage2d_neardup_metrics.csv',index=False)

Selesai. Kirim screenshot hasil sel A, B, dan C (dua tabel).